In [0]:
%run ./01_config

In [0]:
"""
15_robustness.py  —  Robustness: independent generation, misspecification and real-world factors

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 15.
"""

# 15 — Robustness: independent generation, misspecification and real-world factors

# The study's central limitation is that every result comes from one synthetic dataset whose
# generating process belongs to the model family the framework assumes. This notebook
# attacks that limitation directly by re-running the analysis across datasets the framework
# was not built for, and reporting how far the conclusions move.

# Scenario  -  What it tests
# Independent seeds  -  Sampling variability — how much of a result is the seed?
# Lognormal / gamma lifetimes  -  Distributional misspecification: the truth is no longer Weibull
# Seasonality  -  A time-varying hazard the models do not represent
# Operating-condition spread  -  Unobserved duty-cycle heterogeneity
# Multi-cycle maintenance plans  -  MPLA usage more complex than one cycle per item
# Data-quality degradation  -  Missing damage codes, misclassified order types, dropped records
# All combined  -  The four factors Chapter 5 concedes are missing, applied together

# What this notebook is. A reduced-form replication of the pipeline — generate, build
# intervals, fit the pooled AFT, recover per-class parameters, derive intervals — executed
# in memory so that fourteen scenarios run in seconds rather than hours. It omits the
# escalation overlay and the backtest replay, so its absolute values differ from the full
# pipeline's. It is therefore used for relative comparison only: the base scenario is
# the reference, and every conclusion below is a movement from that reference, never an
# absolute claim. The base row is reported alongside the full pipeline's figures so the
# reader can see the offset.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import lognorm, gamma as gamma_dist

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"

HORIZON, CUTOFF = 2191, 1461
N_PER_CLASS = 55
MFRS = [f"Mfr-{i}" for i in range(1, 12)]
PLANTS = [f"P{i:02d}" for i in range(1, 7)]
C_TARGET, RECOVERY_TARGET = 0.70, 15.0

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

# Ground truth

# Class-level parameters are taken from the study's own disclosed ground truth, so the
# scenarios perturb the world around the same equipment population rather than inventing a
# different one.

gt = spark.table(tbl("bronze_ground_truth_params")).toPandas().set_index("EQTYP")
for c in ["weibull_beta", "weibull_eta_days", "pm_restoration_rho",
          "pm_cycle_days", "base_preventive_cost"]:
    gt[c] = gt[c].astype(float)
CLASSES = sorted(gt.index)
P = {c: dict(beta=float(gt.loc[c, "weibull_beta"]), eta=float(gt.loc[c, "weibull_eta_days"]),
             rho=float(gt.loc[c, "pm_restoration_rho"]),
             cycle=float(gt.loc[c, "pm_cycle_days"]),
             cp=float(gt.loc[c, "base_preventive_cost"])) for c in CLASSES}
print(f"{len(CLASSES)} classes loaded from disclosed ground truth")

# Lifetime families

# Each family is parameterised to share the Weibull's characteristic life, so a scenario
# changes the *shape* of the hazard rather than its scale. Residual life is drawn from the
# conditional distribution given accumulated virtual age, exactly as in the main generator.

def draw_life(kind, beta, eta, age, rng):
    u = rng.uniform(1e-4, 1 - 1e-4)
    if kind == "weibull":
        a = (age / eta) ** beta if age > 0 else 0.0
        tot = eta * ((a - np.log(1 - u)) ** (1.0 / beta))
    elif kind == "lognormal":
        sigma = 1.0 / beta
        scale = np.exp(np.log(eta) - 0.5 * sigma ** 2)
        Sa = max(1 - lognorm.cdf(age, s=sigma, scale=scale), 1e-12)
        tot = lognorm.ppf(1 - Sa * (1 - u), s=sigma, scale=scale)
    elif kind == "gamma":
        k, scale = beta, eta / beta
        Sa = max(1 - gamma_dist.cdf(age, k, scale=scale), 1e-12)
        tot = gamma_dist.ppf(1 - Sa * (1 - u), k, scale=scale)
    else:
        raise ValueError(kind)
    return max(float(tot) - age, 1.0)

# Scenario simulator

def simulate(seed, kind="weibull", seasonality=0.0, duty_sd=0.0, multi_cycle=False,
             mfr_sd=1.10, plant_sd=0.95, crit_c=0.35, fvar=0.08):
    rng = np.random.default_rng(seed)
    mf = {m: float(rng.normal(0, mfr_sd)) for m in MFRS}
    mf = {k: v - np.mean(list(mf.values())) for k, v in mf.items()}
    pe = {p: float(rng.normal(0, plant_sd)) for p in PLANTS}
    pe = {k: v - np.mean(list(pe.values())) for k, v in pe.items()}
    rows = []
    for c in CLASSES:
        p = P[c]
        for i in range(N_PER_CLASS):
            m = MFRS[int(rng.integers(0, len(MFRS)))]
            pl = PLANTS[int(rng.integers(0, len(PLANTS)))]
            crit = int(rng.integers(1, 5))
            z = float(rng.gamma(1 / fvar, fvar))
            duty = float(rng.normal(0, duty_sd)) if duty_sd else 0.0
            lp = mf[m] + pe[pl] + crit_c * (crit - 2.5) + np.log(z) + duty
            eta_i = p["eta"] * np.exp(-lp / p["beta"])
            cyc = p["cycle"] * (float(rng.choice([0.5, 1.0, 1.0, 2.0])) if multi_cycle else 1.0)
            inst = float(rng.integers(0, 1095))
            va, clock, nxt, pf = 0.0, inst, inst + cyc, 0
            uid = f"{c}-{i}"
            while clock < HORIZON:
                res = draw_life(kind, p["beta"], eta_i, va, rng)
                if seasonality:
                    res *= max(0.2, 1.0 - seasonality * np.sin(2 * np.pi * (clock % 365) / 365))
                fd = clock + res
                if fd <= nxt and fd < HORIZON:
                    rows.append((uid, c, m, pl, crit, clock, fd - clock, 1, 0, pf))
                    pf += 1; va = 0.0; clock = fd; nxt = clock + cyc
                elif nxt < HORIZON:
                    rows.append((uid, c, m, pl, crit, clock, nxt - clock, 0, 1, pf))
                    va = (va + nxt - clock) * (1 - p["rho"]); clock = nxt; nxt = clock + cyc
                else:
                    rows.append((uid, c, m, pl, crit, clock, HORIZON - clock, 0, 0, pf))
                    break
    return pd.DataFrame(rows, columns=["uid", "cls", "mfr", "plant", "crit",
                                       "start", "dur", "evt", "pm", "pf"])

def degrade(d, rng, miss_code=0.30, misclass=0.10, miss_record=0.05):
    """Faults the NIST work-order literature reports as typical of real CMMS data."""
    d = d.copy()
    flip = rng.uniform(size=len(d)) < misclass          # corrective/preventive miscoding
    d.loc[flip, ["evt", "pm"]] = 1 - d.loc[flip, ["evt", "pm"]].values
    d = d[rng.uniform(size=len(d)) >= miss_record].reset_index(drop=True)
    d.loc[rng.uniform(size=len(d)) < miss_code, "crit"] = np.nan   # unpopulated catalog field
    return d

# Estimation, mirroring the pipeline

def virtual_age(d):
    out = np.zeros(len(d))
    dur, evt, pm, cls = d.dur.values, d.evt.values, d.pm.values, d.cls.values
    for _, idx in d.groupby("uid").indices.items():
        va = 0.0
        for i in idx:
            out[i] = va
            rho = P[cls[i]]["rho"]
            va = 0.0 if evt[i] == 1 else ((va + dur[i]) * (1 - rho) if pm[i] == 1 else va + dur[i])
    return out

def concordance(t, e, r):
    t, e, r = np.asarray(t, float), np.asarray(e), np.asarray(r, float)
    c = p = 0.0
    for i in np.where(e == 1)[0]:
        m = t > t[i]
        if not m.any():
            continue
        c += np.sum(r[i] > r[m]) + 0.5 * np.sum(r[i] == r[m]); p += m.sum()
    return c / p if p else np.nan

def design(d, cols=None):
    X = pd.get_dummies(d[["mfr", "plant", "cls"]], drop_first=True).astype(float)
    X["crit"] = d.crit.fillna(2.5).values - 2.5
    X["va"] = d["_va"].values / 500.0
    X["pf"] = d.pf.values
    return X if cols is None else X.reindex(columns=cols, fill_value=0.0)

def fit_aft(X, t, e, v):
    X = np.asarray(X, float); k = X.shape[1]
    def nll(q):
        b, eta = np.exp(q[0]), np.exp(q[1])
        ei = eta * np.exp(-(X @ q[2:]) / b)
        x1, x0 = (v + t) / ei, v / ei
        return -(np.sum(e * (np.log(b / ei) + (b - 1) * np.log(np.maximum(x1, 1e-9))))
                 - np.sum(x1 ** b - x0 ** b))
    r = minimize(nll, np.r_[np.log(2), np.log(600), np.zeros(k)],
                 method="L-BFGS-B", options={"maxiter": 3000})
    return float(np.exp(r.x[0])), float(np.exp(r.x[1])), r.x[2:]

def fit_offset(t, e, v, off):
    def nll(q):
        b, eta = np.exp(q)
        ei = eta * np.exp(-off / b)
        x1, x0 = (v + t) / ei, v / ei
        return -(np.sum(e * (np.log(b / ei) + (b - 1) * np.log(np.maximum(x1, 1e-9))))
                 - np.sum(x1 ** b - x0 ** b))
    r = minimize(nll, [np.log(2), np.log(600)], method="Nelder-Mead",
                 options={"maxiter": 4000})
    return np.exp(r.x)

def cost_rate(T, b, eta, cp, cf):
    g = np.linspace(0, T, 300)
    ST = float(np.exp(-(T / eta) ** b))
    return (cp * ST + cf * (1 - ST)) / max(float(np.trapezoid(np.exp(-(g / eta) ** b), g)), 1e-9)

def optimise(b, eta, cp, cf):
    Ts = np.linspace(20, eta * 3, 200)
    return float(Ts[int(np.argmin([cost_rate(t, b, eta, cp, cf) for t in Ts]))])

def run(label, seed=42, degrade_data=False, **kw):
    d = simulate(seed, **kw)
    if degrade_data:
        d = degrade(d, np.random.default_rng(seed + 1))
    d = d.sort_values(["uid", "start"]).reset_index(drop=True)
    d["_va"] = virtual_age(d)
    tr, te = d[d.start <= CUTOFF], d[d.start > CUTOFF]
    if tr.evt.sum() < 50 or te.evt.sum() < 20:
        print(f"skip {label}: too few events"); return None
    Xtr = design(tr); Xte = design(te, Xtr.columns)
    b, eta, g = fit_aft(Xtr.values, tr.dur.values.astype(float),
                        tr.evt.values.astype(float), tr._va.values)
    C = concordance(te.dur, te.evt, Xte.values @ g)
    cov = [i for i, c in enumerate(Xtr.columns) if not c.startswith("cls_") and c != "va"]
    om = float((Xtr.values[:, cov] @ g[cov]).mean())
    berr, eerr, sav, wts = [], [], [], []
    for c in CLASSES:
        gtr = tr[tr.cls == c]
        if gtr.evt.sum() < 10:
            continue
        off = design(gtr, Xtr.columns).values[:, cov] @ g[cov] - om
        bh, eh = fit_offset(gtr.dur.values.astype(float), gtr.evt.values.astype(float),
                            gtr._va.values, off)
        bt, et = P[c]["beta"], P[c]["eta"]
        berr.append(100 * abs(bh - bt) / bt); eerr.append(100 * abs(eh - et) / et)
        cp, cf = P[c]["cp"], P[c]["cp"] * 4.0
        base = cost_rate(P[c]["cycle"], bt, et, cp, cf)
        sav.append(100 * (base - cost_rate(optimise(bh, eh, cp, cf), bt, et, cp, cf)) / base)
        wts.append(base)
    return {"scenario": label, "intervals": len(d), "events": int(d.evt.sum()),
            "concordance": round(C, 4),
            "beta_err_median_pct": round(float(np.median(berr)), 1),
            "eta_err_median_pct": round(float(np.median(eerr)), 1),
            "rq2_saving_pct": round(float(np.average(sav, weights=wts)), 2)}

# Run the scenario set

SEEDS = [101, 202, 303, 404, 505]
out = [run("S0 base (reference)")]
for s in SEEDS:
    out.append(run(f"S1 independent seed {s}", seed=s))
out += [
    run("S2 lognormal lifetimes", kind="lognormal"),
    run("S3 gamma lifetimes", kind="gamma"),
    run("S4 seasonality (+/-30% hazard)", seasonality=0.30),
    run("S5 operating-condition spread", duty_sd=0.60),
    run("S6 multi-cycle maintenance plans", multi_cycle=True),
    run("S7 data-quality degradation", degrade_data=True),
    run("S8 all real-world factors combined", kind="lognormal", seasonality=0.30,
        duty_sd=0.60, multi_cycle=True, degrade_data=True),
]
rob = pd.DataFrame([o for o in out if o])
display(spark.createDataFrame(rob))

# Interpretation

base = rob.iloc[0]
seeds = rob[rob.scenario.str.startswith("S1")]
print(f"Reference scenario: C={base.concordance}, shape err {base.beta_err_median_pct}%, "
      f"RQ2 {base.rq2_saving_pct}%")
print(f"\nSampling variability across {len(seeds)} independent seeds:")
print(f"  concordance   {seeds.concordance.min():.3f} to {seeds.concordance.max():.3f}")
print(f"  shape error   {seeds.beta_err_median_pct.min():.1f}% to "
      f"{seeds.beta_err_median_pct.max():.1f}%")
print(f"  RQ2 saving    {seeds.rq2_saving_pct.min():.2f}% to {seeds.rq2_saving_pct.max():.2f}%")
print("\nThe RQ2 spread across seeds is the single most important number here: it bounds how "
      "much of the headline interval-optimisation result is attributable to the particular "
      "dataset realisation rather than to the method.")

stress = rob[rob.scenario.str.match(r"S[2-8]")]
worst = stress.loc[stress.concordance.idxmin()]
print(f"\nWorst discrimination under stress: {worst.scenario} at C={worst.concordance} "
      f"({'below' if worst.concordance < C_TARGET else 'still above'} the {C_TARGET} criterion)")
misspec = rob[rob.scenario.str.match(r"S[23]")]
print(f"Distributional misspecification: concordance "
      f"{misspec.concordance.min():.3f}-{misspec.concordance.max():.3f}, but shape error rises "
      f"to {misspec.beta_err_median_pct.max():.1f}% — discrimination survives a wrong lifetime "
      f"family; parameter recovery does not.")

# Figure — sensitivity of each headline to each scenario

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6), sharey=True)
metrics = [("concordance", "concordance", C_TARGET, ACCENT),
           ("beta_err_median_pct", "median shape error (%)", RECOVERY_TARGET, WARM),
           ("rq2_saving_pct", "RQ2 realised saving (%)", None, GREEN)]
y = np.arange(len(rob))[::-1]
for ax, (col, label, ref, colour) in zip(axes, metrics):
    ax.barh(y, rob[col], color=[MUTED if i == 0 else colour for i in range(len(rob))], height=.62)
    ax.axvline(float(base[col]), color=INK, ls=":", lw=1.1)
    if ref is not None:
        ax.axvline(ref, color=WARM if col == "concordance" else INK, ls="--", lw=1.2)
    ax.set_xlabel(label)
axes[0].set_yticks(y); axes[0].set_yticklabels(rob.scenario, fontsize=7.5)
fig.suptitle("dotted = reference scenario    dashed = pre-registered criterion",
             fontsize=8, color=MUTED, y=1.02)
emit(fig, "r1_robustness_scenarios")

# Persist

from pyspark.sql.functions import lit

(spark.createDataFrame(rob).withColumn("dataset_version", lit(DATASET_VERSION))
      .write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(tbl("results_robustness")))
rob.to_csv(f"{EXPORTS}/results_robustness.csv", index=False)
print("written: results_robustness")
print("\nFor the external-validity section: quote the seed spread as sampling variability, the "
      "misspecification rows as evidence that discrimination is robust while recovery is not, "
      "and the data-quality row as the closest available proxy for real CMMS conditions.")